In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

figure_dir = "figures/revision/supplement-extra"
setup_plotting(figure_dir, display_dpi=200, save_dpi=300)

import os

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

from spatial_tcr.spatial import (
    fill_annotations,
    spatial_expansion,
    spatial_sample_split,
)

## Load data

In [ ]:
data_dir = "data/xenium/processed"
path = f"{data_dir}/08.1-kidney_tcr_clonal_clusters.h5ad"
adata = sc.read_h5ad(path)

# remove control samples
adata = adata[adata.obs["condition"] == "ANCA"].copy()
adata

In [ ]:
adata.obs["glom_annot"].value_counts()

## Annotate Periglom. areas.

In [ ]:
spatial_sample_split(adata, sample_key="sample", out_key="spatial")

In [ ]:
sc.pl.spatial(adata, color="sample", spot_size=10)

In [ ]:
fill_annotations(
    adata,
    obs_key="glom_annot",
    out_key="glom_annot_filled",
    sample_key="sample",
    spatial_key="spatial",
)

In [ ]:
ad_sub = adata[adata.obs["sample"] == adata.obs["sample"].unique()[0]]
sc.pl.spatial(ad_sub, color="glom_annot_filled", spot_size=10)

In [ ]:
spatial_expansion(
    adata,
    obs_key="glom_annot_filled",
    out_key="glom_annot_filled_expanded",
    expansion=100,
    sample_key="sample",
    spatial_key="spatial",
)

In [ ]:
ad_sub = adata[adata.obs["sample"] == adata.obs["sample"].unique()[0]]
sc.pl.spatial(ad_sub, color="glom_annot_filled", spot_size=10, title="Glom. Annotation")
sc.pl.spatial(ad_sub, color="glom_annot_filled_expanded", spot_size=10)

In [ ]:
adata.obs["peri_glom_annot"] = adata.obs["glom_annot_filled_expanded"].copy()
adata.obs.loc[adata.obs["glom_annot_filled"].notna(), "peri_glom_annot"] = pd.NA


ad_sub = adata[adata.obs["sample"] == adata.obs["sample"].unique()[0]]

import matplotlib as mpl

mpl.rcParams["figure.figsize"] = (3.5, 3.5)

fig = sc.pl.spatial(
    ad_sub,
    color="peri_glom_annot",
    spot_size=10,
    title="Periglom. Annotation",
    return_fig=True,
    show=False,
)
ax = fig.axes[0]

# x/y label fontsize
ax.set_xlabel(ax.get_xlabel(), fontsize=8)
ax.set_ylabel(ax.get_ylabel(), fontsize=8)

# title fontsize
ax.set_title(ax.get_title(), fontsize=10)

# legend fontsize (if present)
leg = ax.get_legend()
if leg is not None:
    for t in leg.get_texts():
        t.set_fontsize(10)
    if leg.get_title() is not None:
        leg.get_title().set_fontsize(10)

fig.savefig(
    os.path.join(figure_dir, "peri_glom_annot.pdf"), bbox_inches="tight", dpi=300
)

In [ ]:
mpl.rcParams["figure.figsize"] = (3.5, 3.5)
fig = sc.pl.spatial(
    ad_sub,
    color="glom_annot_filled",
    spot_size=10,
    title="Glom. Annotation",
    return_fig=True,
    show=False,
)
ax = fig.axes[0]

# x/y label fontsize
ax.set_xlabel(ax.get_xlabel(), fontsize=8)
ax.set_ylabel(ax.get_ylabel(), fontsize=8)

# title fontsize
ax.set_title(ax.get_title(), fontsize=10)

# legend fontsize (if present)
leg = ax.get_legend()
if leg is not None:
    for t in leg.get_texts():
        t.set_fontsize(10)
    if leg.get_title() is not None:
        leg.get_title().set_fontsize(10)
plt.savefig(os.path.join(figure_dir, "glom_annot_filled.pdf"))

In [ ]:
adata.obs["in_peri_glom"] = adata.obs["peri_glom_annot"].notna()

## Save annotations

In [ ]:
data_dir = "data/xenium/processed"
path = f"{data_dir}/08.2-kidney_tcr_clonal_clusters_peri_glom_annot.h5ad"
adata.write_h5ad(path)